# Espelho Meu | modelo de maquiagem

## Problema e origem dos dados

Classificar um estilo de acabamento com base em tipo de pele, ocasião e preferência declarados. O resultado é uma classe textual oferecida na consultoria. A entrada é o questionário.

Os perfis resultam do produto cartesiano de 3 tipos de pele, 3 ocasiões e 3 acabamentos. Os rótulos foram atribuídos por heurísticas aplicadas em ordem: matte se escolhido ou para pele oleosa em evento; luminoso se escolhido ou para pele seca em evento; natural nos demais casos.

A base foi gerada para a prova de conceito. Não contém respostas coletadas de pessoas, avaliações profissionais nem fotografias. O teste abaixo mede a reprodução das heurísticas em combinações reservadas.

In [1]:
from pathlib import Path
import sys

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent
sys.path.insert(0, str(raiz))

import joblib
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

from configuracao import ALVO, CAMPOS
from treinar import carregar_dados

area = 'maquiagem'
arquivo = raiz / "dados" / f"{area}.csv"
bruto = pd.read_csv(arquivo, dtype=str)
campos = CAMPOS[area]

## Análise exploratória e preparação

Inspecione o tamanho, os valores ausentes, a duplicação e o equilíbrio das classes. A rotina compartilhada valida as categorias, elimina linhas idênticas e rejeita perfis repetidos com rótulos conflitantes. A frequência por classe ajuda a interpretar a acurácia e o F1 macro.

In [2]:
print(f"{len(bruto)} perfis | {len(bruto.columns)} colunas")
print("Ausentes por coluna:")
print(bruto.isna().sum().to_string())
print(f"Linhas duplicadas: {bruto.duplicated().sum()}")
dados = carregar_dados(area, raiz / "dados")
print("\nAmostra:")
print(dados.head().to_string(index=False))
print("\nDistribuição das classes:")
print(dados[ALVO].value_counts().to_string())
print("\nDistribuição das entradas:")
for nome in campos:
    print(nome, dados[nome].value_counts().to_dict())

27 perfis | 4 colunas
Ausentes por coluna:
tipo_pele       0
ocasiao         0
acabamento      0
recomendacao    0
Linhas duplicadas: 0

Amostra:
tipo_pele   ocasiao acabamento       recomendacao
     seca dia_a_dia    natural  maquiagem natural
     seca dia_a_dia      matte    maquiagem matte
     seca dia_a_dia   luminoso maquiagem luminosa
     seca  trabalho    natural  maquiagem natural
     seca  trabalho      matte    maquiagem matte

Distribuição das classes:
recomendacao
maquiagem matte       11
maquiagem luminosa     9
maquiagem natural      7

Distribuição das entradas:
tipo_pele {'seca': 9, 'mista': 9, 'oleosa': 9}
ocasiao {'dia_a_dia': 9, 'trabalho': 9, 'evento': 9}
acabamento {'natural': 9, 'matte': 9, 'luminoso': 9}


## Treinamento

A saída categórica pede classificação. A árvore de decisão permite aprender combinações das respostas sem pressupor ordem entre as categorias codificadas. A profundidade 4 limita a complexidade. A divisão estratificada reserva 25% dos perfis para teste e fixa a semente 42 para permitir a reprodução.

O baseline sempre prevê a classe mais frequente dos dados de treino. As categorias possíveis vêm do contrato do questionário, e a árvore é ajustada somente no conjunto de treino.

In [3]:
X = dados[list(campos)]
y = dados[ALVO]
x_treino, x_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

modelo = make_pipeline(
    OneHotEncoder(categories=list(campos.values())),
    DecisionTreeClassifier(max_depth=4, random_state=42),
)
modelo.fit(x_treino, y_treino)
baseline = DummyClassifier(strategy="most_frequent").fit(x_treino, y_treino)
print(f"Treino: {len(x_treino)} perfis | Teste: {len(x_teste)} perfis")

Treino: 20 perfis | Teste: 7 perfis


## Avaliação no teste

A acurácia mede a fração de acertos. O F1 macro dá o mesmo peso a cada classe. Na matriz de confusão, a linha é o rótulo esperado e a coluna é a predição. O suporte registra a quantidade de exemplos de teste por classe.

In [4]:
previsoes = modelo.predict(x_teste)
print(f"Acurácia: {accuracy_score(y_teste, previsoes):.2%}")
print(f"F1 macro: {f1_score(y_teste, previsoes, average='macro'):.2%}")
print(f"Acurácia do baseline: {accuracy_score(y_teste, baseline.predict(x_teste)):.2%}")
print(classification_report(y_teste, previsoes, zero_division=0))
rotulos = modelo.classes_
matriz = pd.DataFrame(
    confusion_matrix(y_teste, previsoes, labels=rotulos),
    index=pd.Index(rotulos, name="esperado"),
    columns=pd.Index(rotulos, name="previsto"),
)
print(matriz.to_string())

Acurácia: 71.43%
F1 macro: 58.33%
Acurácia do baseline: 42.86%
                    precision    recall  f1-score   support

maquiagem luminosa       0.00      0.00      0.00         2
   maquiagem matte       0.60      1.00      0.75         3
 maquiagem natural       1.00      1.00      1.00         2

          accuracy                           0.71         7
         macro avg       0.53      0.67      0.58         7
      weighted avg       0.54      0.71      0.61         7

previsto            maquiagem luminosa  maquiagem matte  maquiagem natural
esperado                                                                  
maquiagem luminosa                   0                2                  0
maquiagem matte                      0                3                  0
maquiagem natural                    0                0                  2


Nenhum dos dois exemplos de maquiagem luminosa foi classificado corretamente. O teste é pequeno e herda as regras do CSV. Reúna perfis rotulados e revisados de forma independente para avaliar adequação das sugestões. Evite usar este resultado como evidência de eficácia em pessoas reais.

## Exportação para a API

Exporte o pipeline completo com o pré-processamento e o classificador. O Flask carrega o arquivo da área na inicialização.

In [5]:
pasta_modelos = raiz / "modelos"
pasta_modelos.mkdir(exist_ok=True)
saida = pasta_modelos / f"{area}.joblib"
joblib.dump(modelo, saida)
recarregado = joblib.load(saida)
assert list(recarregado.predict(x_teste)) == list(previsoes)
print(f"Exportado e validado: modelos/{saida.name}")

Exportado e validado: modelos/maquiagem.joblib
